# Q1: Inter-order Gap

Tính trung vị số ngày giữa hai lần mua liên tiếp của khách hàng có hơn 1 đơn hàng.

In [ ]:
import pandas as pd

# string -> datetime
orders = pd.read_csv("data/raw/orders.csv", parse_dates=["order_date"])

# sort ASC
orders = orders.sort_values(["customer_id", "order_date"])

# calculate the gap ()
orders["prev_date"] = orders.groupby("customer_id")["order_date"].shift(1)
orders["gap_days"] = (orders["order_date"] - orders["prev_date"]).dt.days

# bỏ NaN
gaps = orders["gap_days"].dropna()

# lọc realistic range 
gaps = gaps[(gaps > 0) & (gaps < 365)]

print("Median gap:", gaps.median())

**Answer: (B).90 ngày**

# Q2: Average gross profit margin

Phân khúc sản phẩm nào trong product.csv có tỷ suất lợi nhuận trung bình gộp cao nhất, theo công thức (price - cogs)/price ?

In [ ]:
import pandas as pd

product = pd.read_csv("data/raw/products.csv")

# condition
product = product[(product["price"] > 0) & (product["cogs"] < product["price"])]

# calculate margin
product["margin"] = (product["price"] - product["cogs"]) / product["price"]

# group by segment
segment_margin = product.groupby("segment")["margin"].mean()

# find max
best_segment = segment_margin.idxmax()

print(segment_margin.sort_values(ascending=False))
print("\nBest segment:", best_segment)

**Answer: (D).Standard**

# Q3: Max reason return product

Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

In [ ]:
import pandas as pd

# read file csv
products = pd.read_csv("data/raw/products.csv")
returns = pd.read_csv("data/raw/returns.csv")

# join with product_id
# use inner join, just taking the relational data
merged_product = pd.merge(returns, products, on="product_id")

# find product of 'Streetwear'
streetwear_returns = merged_product[merged_product["category"] == "Streetwear"]

# numbers of reason
reason_counts = streetwear_returns["return_reason"].value_counts()

# find max reason
most_common_reason = reason_counts.idxmax()

print("Reason return in Streetwear segment:")
print(reason_counts)
print("-" * 30)
print(f"Correct Answer: {most_common_reason}")

**Answer: (B).wrong-size**

# Q4: Lowest bounce-rate

Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

In [ ]:
import pandas as pd

# Load the web traffic dataset
find_lowest = pd.read_csv("data/raw/web_traffic.csv")

# Group by traffic source and calculate the mean bounce rate
bounce_analysis = find_lowest.groupby("traffic_source")["bounce_rate"].mean().sort_values()

# Display the results
print("Average Bounce Rate by Source:")
print(bounce_analysis)

# Identify the lowest
best_source = bounce_analysis.idxmin()
print(f"\nThe source with the lowest average bounce rate is: {best_source}")

**Answer: (C).email-campaign**

# Q5: Percentage promo_id

Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?

In [ ]:
import pandas as pd

# read csv file
dataset = pd.read_csv("data/raw/order_items.csv")

# .count() ignores null
applied_promo_count = dataset["promo_id"].count()

# count the total row
total_rows = len(dataset)

# calculate the percentage
promo_percentage = (applied_promo_count / total_rows) * 100

# resuslt
print(f"Total items: {total_rows}")
print(f"Items with promo_id: {applied_promo_count}")
print(f"Percentage of rows with promotion: {round(promo_percentage)}%")

**Answer: (C).39%**

# Q6: Average order per customer

Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)

In [ ]:
import pandas as pd

# read data
customers = pd.read_csv("data/raw/customers.csv")
orders = pd.read_csv("data/raw/orders.csv")

# choose customers with not null value
customers = customers.dropna(subset=["age_group"])

# join two table - inner join
df_merged = pd.merge(orders, customers, on="customer_id")

# calculate amount of orders and customers per age
age_stats = df_merged.groupby("age_group").agg(
    total_orders=('order_id', 'count'),
    total_customers=('customer_id', 'nunique')
)

# average amount of orders
age_stats["avg_orders_per_customer"] = age_stats["total_orders"] / age_stats["total_customers"]

# sort
result = age_stats["avg_orders_per_customer"].sort_values(ascending=False)

print(result)

**Answer: (A).55+**

# Q7: Highest total revenue by region

Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong
sales_train.csv?

In [ ]:
import pandas as pd

# read data
sales_train = pd.read_csv("data/raw/sales.csv") 
geo = pd.read_csv("data/raw/geography.csv")
orders = pd.read_csv("data/raw/orders.csv")
customers = pd.read_csv("data/raw/customers.csv")

# link depeand on date (sales.csv dont have zip)
sales_train["Date"] = pd.to_datetime(sales_train["Date"])
orders["order_date"] = pd.to_datetime(orders["order_date"])

# join table
data_step1 = pd.merge(sales_train, orders, left_on="Date", right_on="order_date")

# if there are two zip - change the name to zip_x, zip_y
data_step2 = pd.merge(data_step1, customers, on="customer_id")

# depend on zipx,y join the final result
data_final = pd.merge(data_step2, geo, left_on="zip_x", right_on="zip")

# calculate highest revenue region
region_revenue = data_final.groupby("region")["Revenue"].sum().sort_values(ascending=False)

print("Total revenue per region:")
print(region_revenue)

# Answer
best_region = region_revenue.idxmax()
print("-" * 30)
print(f"The region with highest revenue: {best_region}")

**Answer: (C).East**

# Q8: Highest payment methods

Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức
thanh toán nào được sử dụng nhiều nhất?

In [ ]:
import pandas as pd

# read file csv
orders = pd.read_csv("data/raw/orders.csv")

# choose the exist status
cancelled_orders = orders[orders["order_status"] == "cancelled"]

# count the numbers of payment
payment_counts = cancelled_orders["payment_method"].value_counts()

# show raking
print("Payment methods for cancelled orders:")
print(payment_counts)

# show result
most_common_cancelled_payment = payment_counts.idxmax()
print("-" * 30)
print(f"The answer is: {most_common_cancelled_payment}")

**Answer: (A).credit_card**

# Q9: Highest return rate by maximum size

Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?

In [ ]:
import pandas as pd

# load data from csv file
products = pd.read_csv("data/raw/products.csv")
order_items = pd.read_csv("data/raw/order_items.csv")
returns = pd.read_csv("data/raw/returns.csv")

# count size of table merge by products and order_items
orders_with_size = pd.merge(order_items, products[['product_id', 'size']], on='product_id')
total_orders_per_size = orders_with_size.groupby('size').size()

# count size of table merge by products and returns
returns_with_size = pd.merge(returns, products[['product_id', 'size']], on='product_id')
total_returns_per_size = returns_with_size.groupby('size').size()

# calculate return rate of (S, M, L, XL)
return_rate = total_returns_per_size / total_orders_per_size

# result
print("Return Rate by Size:")
print(return_rate.sort_values(ascending=False))

# Answer
highest_rate_size = return_rate.idxmax()
print("-" * 30)
print(f"The size with the highest return rate is: {highest_rate_size}")

**Answer: (A).S**

# Q10: Which installment plan has the average payment value per order?

Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên
mỗi đơn hàng cao nhất?

In [ ]:
import pandas as pd

# read csv file
payments = pd.read_csv("data/raw/payments.csv")

# group by installments and payment_value to calculate
avg_payment_by_plan = payments.groupby("installments")["payment_value"].mean()

# approximately the value to fit with nums of installments
plans_of_interest = [1, 3, 6, 12]
result = avg_payment_by_plan.reindex(plans_of_interest)

# sort
print("Average Payment Value per installment plan:")
print(result.sort_values(ascending=False))

# answer
best_plan = result.idxmax()
print("-" * 30)
print(f"The installment plan with the highest average payment is: {best_plan} installments")

**Answer: (C).6**